In [5]:
# Install packages
!pip install -q streamlit pandas plotly

from pathlib import Path
import subprocess
import sys
import time
import webbrowser

# Create the Streamlit application
app_code = '''
from pathlib import Path
import streamlit as st
import pandas as pd
import plotly.express as px

st.set_page_config(
    page_title="World Happiness Dashboard",
    page_icon="🌍",
    layout="wide"
)

# Find the CSV file
app_folder = Path(__file__).resolve().parent

possible_paths = [
    app_folder.parent / "data" / "world_happiness_2023.csv",
    app_folder / "data" / "world_happiness_2023.csv",
    app_folder / "world_happiness_2023.csv"
]

data_path = next(
    (path for path in possible_paths if path.exists()),
    None
)

if data_path is None:
    st.error("world_happiness_2023.csv was not found.")
    st.stop()

# Load data
df = pd.read_csv(data_path)

df.columns = [
    "Country",
    "Region",
    "Score",
    "GDP",
    "Social_Support",
    "Life_Expectancy",
    "Freedom",
    "Generosity",
    "Corruption"
]

# Title
st.title("🌍 World Happiness Dashboard")

# Sidebar filters
regions = ["All"] + sorted(df["Region"].unique().tolist())

selected_region = st.sidebar.selectbox(
    "Select Region",
    regions
)

top_n = st.sidebar.slider(
    "Show Top Countries",
    5,
    25,
    15
)

# Filter data
if selected_region == "All":
    filtered = df.copy()
else:
    filtered = df[df["Region"] == selected_region].copy()

# KPI cards
col1, col2, col3 = st.columns(3)

col1.metric("Countries", len(filtered))

col2.metric(
    "Average Score",
    f"{filtered['Score'].mean():.2f}"
)

happiest_country = filtered.loc[
    filtered["Score"].idxmax(),
    "Country"
]

col3.metric(
    "Happiest Country",
    happiest_country
)

# Chart 1
st.subheader("Happiness Rankings")

top_countries = (
    filtered.nlargest(top_n, "Score")
    .sort_values("Score")
)

fig1 = px.bar(
    top_countries,
    x="Score",
    y="Country",
    orientation="h",
    color="Score",
    color_continuous_scale="Blues"
)

st.plotly_chart(fig1, width="stretch")

# Chart 2
st.subheader("GDP vs Happiness")

fig2 = px.scatter(
    filtered,
    x="GDP",
    y="Score",
    hover_name="Country"
)

st.plotly_chart(fig2, width="stretch")

# Chart 3: Lecture 09 exercise
st.subheader("Difference from Global Average")

global_average = df["Score"].mean()

filtered["Difference"] = (
    filtered["Score"] - global_average
)

filtered["Absolute_Difference"] = (
    filtered["Difference"].abs()
)

difference_data = (
    filtered.nlargest(top_n, "Absolute_Difference")
    .sort_values("Difference")
)

fig3 = px.bar(
    difference_data,
    x="Difference",
    y="Country",
    orientation="h",
    color="Difference",
    color_continuous_scale="RdBu",
    color_continuous_midpoint=0
)

fig3.add_vline(
    x=0,
    line_dash="dash",
    annotation_text=f"Global Average: {global_average:.2f}"
)

st.plotly_chart(fig3, width="stretch")
'''

# Save the app
Path("lecture09exercise.py").write_text(
    app_code,
    encoding="utf-8"
)

# Start Streamlit
subprocess.Popen([
    sys.executable,
    "-m",
    "streamlit",
    "run",
    "lecture09exercise.py",
    "--server.port",
    "8501"
])

time.sleep(4)

# Open the website
url = "http://localhost:8501"
print("Streamlit website:", url)
webbrowser.open(url)


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Streamlit website: http://localhost:8501


True